# Embeddings -- Build: Fine-Tuned Embedding Model + Evaluation

> **Section 01 -- Topic 02 -- build** | Prerequisites: All embeddings notebooks (`foundations.ipynb`, `advanced.ipynb`, `expert.ipynb`)

This capstone notebook brings together everything from the embeddings series into a single,
end-to-end project. We will choose a domain (technical documentation), create training data,
fine-tune a sentence-transformer with MNRL loss, evaluate on three standard tasks (retrieval,
semantic textual similarity, and clustering), visualize the embedding space before and after
fine-tuning, quantize the model for deployment, and produce a model card.

This is the pattern you will follow in production: domain analysis, data curation, training,
rigorous evaluation, quantization, and documentation. Every step is here with real, runnable code.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import (
    adjusted_rand_score, normalized_mutual_info_score,
    silhouette_score, v_measure_score
)
from scipy.stats import spearmanr
from sentence_transformers import (
    SentenceTransformer, InputExample, losses, evaluation
)
from torch.utils.data import DataLoader
import json
import time
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print('All imports ready. Starting build project.')

---
## 1. Define Domain and Collect Data

### 1.1 Domain Analysis

We are building an embedding model for **technical infrastructure documentation** -- the kind of
content found in internal wikis, runbooks, and configuration guides at technology companies. This
domain has several characteristics that make general-purpose embeddings suboptimal:

- **Specialized vocabulary**: Terms like "sidecar proxy," "blue-green deployment," and "circuit
  breaker pattern" carry precise technical meaning that general models may not fully capture.
- **Query-document asymmetry**: Users ask short, natural-language questions ("how do I restart the
  cache?") that should match verbose technical paragraphs.
- **Synonymy and abbreviation**: "k8s" means "Kubernetes," "LB" means "load balancer," and "TLS"
  and "SSL" are used interchangeably in practice.

A fine-tuned model should learn these domain-specific relationships, improving retrieval accuracy
for technical queries without sacrificing general language understanding.

In [ ]:
# Define training data: (query, relevant_passage) pairs
# In production, these would come from click logs, QA pairs, or LLM-generated queries

training_pairs = [
    # Kubernetes / Container Orchestration
    ('How to scale pods in Kubernetes', 'Use kubectl scale deployment/<name> --replicas=N or configure a HorizontalPodAutoscaler with target CPU utilization in the deployment spec.'),
    ('k8s pod keeps crashing', 'Check pod logs with kubectl logs <pod-name> and describe the pod with kubectl describe pod <pod-name>. Common causes include OOMKilled (increase memory limits), CrashLoopBackOff (fix application error), and ImagePullBackOff (verify image name and registry credentials).'),
    ('container resource limits', 'Set resources.requests and resources.limits in the container spec. Requests are used for scheduling decisions while limits are enforced by the kubelet. CPU is measured in millicores and memory in bytes (Mi/Gi).'),
    ('rolling update strategy', 'Configure strategy.type: RollingUpdate with maxSurge and maxUnavailable parameters. MaxSurge controls how many extra pods can be created during the update, while maxUnavailable controls how many pods can be down simultaneously.'),
    ('kubernetes service mesh setup', 'Install Istio or Linkerd as a service mesh. The mesh injects sidecar proxies into each pod that handle mTLS, load balancing, retries, and observability without application code changes.'),
    ('how to debug k8s networking', 'Use kubectl exec to run network diagnostics from within pods. Check Service endpoints, NetworkPolicy rules, and DNS resolution with nslookup. Tools like tcpdump and netcat help diagnose connectivity issues between services.'),
    
    # Database Administration
    ('optimize slow database queries', 'Run EXPLAIN ANALYZE on slow queries to examine the execution plan. Add indexes on columns used in WHERE, JOIN, and ORDER BY clauses. Consider partial indexes for queries with common filter predicates. Monitor pg_stat_statements for query performance trends.'),
    ('database connection pool configuration', 'Set pool_size to match your expected concurrent connections. Configure max_overflow for burst traffic. Set pool_timeout to fail fast rather than queue indefinitely. Use pool_recycle to prevent stale connections from long-lived server processes.'),
    ('database replication setup', 'Configure streaming replication by setting wal_level=replica on the primary and creating a replication slot. On the replica, use pg_basebackup to initialize and set primary_conninfo in recovery.conf to point to the primary server.'),
    ('how to handle database migrations safely', 'Use a migration tool like Alembic or Flyway. Write backward-compatible migrations that can be rolled back. For large tables, use CREATE INDEX CONCURRENTLY to avoid locking. Deploy migrations separately from application code for zero-downtime releases.'),
    ('database backup and restore', 'Schedule automated backups using pg_dump for logical backups or pg_basebackup for physical backups. Store backups in an offsite location with encryption. Test restore procedures regularly by restoring to a staging environment.'),
    
    # Networking and Security
    ('SSL certificate renewal process', 'Certificates from Lets Encrypt auto-renew via certbot. For manual certs, generate a new CSR with openssl req, submit to your CA, and install the new certificate. Update the TLS secret in Kubernetes with kubectl create secret tls.'),
    ('configure firewall rules', 'Define inbound and outbound rules using security groups or iptables. Use CIDR notation for IP ranges. Follow the principle of least privilege: deny all by default and allow only required traffic on specific ports.'),
    ('load balancer health check setup', 'Configure the health check endpoint path (typically /health), interval (10-30s), timeout (5s), and unhealthy threshold (3 consecutive failures). The health endpoint should verify critical dependencies like database connectivity.'),
    ('VPN configuration for remote access', 'Set up WireGuard or OpenVPN server with certificate-based authentication. Configure split tunneling to route only corporate traffic through the VPN. Distribute client configurations through a secure channel.'),
    ('DDoS mitigation strategy', 'Deploy rate limiting at the edge with tools like Cloudflare or AWS Shield. Configure connection limits per IP. Use challenge-response (CAPTCHA) for suspicious traffic patterns. Set up geographic blocking for regions with no legitimate users.'),
    
    # Monitoring and Observability
    ('set up prometheus alerting', 'Define alerting rules in Prometheus using PromQL expressions. Configure AlertManager with routing rules to send notifications to Slack, PagerDuty, or email based on severity labels. Set appropriate for: durations to avoid flapping.'),
    ('distributed tracing configuration', 'Instrument services with OpenTelemetry SDK. Configure the OTLP exporter to send traces to Jaeger or Zipkin. Propagate trace context headers (traceparent) across service boundaries. Set appropriate sampling rates to control overhead.'),
    ('log aggregation pipeline', 'Deploy Fluentd or Vector as a log shipper on each node. Parse and structure logs into JSON format. Route logs to Elasticsearch or Loki for storage and querying. Set retention policies based on log volume and compliance requirements.'),
    ('SLO and error budget monitoring', 'Define Service Level Objectives as a percentage of good requests over a rolling window (e.g., 99.9% success rate over 30 days). Calculate error budget burn rate and set alerts when the budget is being consumed faster than expected.'),
    
    # CI/CD and Deployment
    ('blue green deployment setup', 'Maintain two identical production environments (blue and green). Deploy new version to the inactive environment, run smoke tests, then switch the load balancer to point to the new environment. Keep the old environment as instant rollback.'),
    ('canary deployment configuration', 'Route a small percentage (1-5%) of traffic to the new version using weighted routing. Monitor error rates, latency, and business metrics. Gradually increase traffic if metrics are healthy. Automated rollback if error rate exceeds threshold.'),
    ('CI pipeline optimization', 'Cache dependencies between builds. Parallelize test suites across multiple runners. Use incremental builds that only recompile changed modules. Set up test splitting based on historical timing data.'),
    ('GitOps workflow setup', 'Store all infrastructure and application configuration in Git. Use ArgoCD or Flux to automatically sync cluster state with the Git repository. Changes go through pull request review before being applied to the cluster.'),
    ('artifact registry management', 'Use a private registry like Harbor or ECR. Tag images with both the Git SHA and semantic version. Set up vulnerability scanning on push. Configure garbage collection to remove untagged images older than the retention period.'),
    
    # Infrastructure as Code
    ('terraform state management', 'Store Terraform state in a remote backend (S3 + DynamoDB for locking). Never edit state manually. Use terraform import to bring existing resources under management. Enable state encryption at rest.'),
    ('infrastructure drift detection', 'Run terraform plan on a schedule to detect drift between declared and actual infrastructure. Alert on any detected changes. Investigate and either update the code to match reality or apply the code to fix the drift.'),
    ('secrets management in infrastructure code', 'Never commit secrets to version control. Use a secrets manager (Vault, AWS Secrets Manager) and reference secrets by path. Inject secrets as environment variables at deployment time. Rotate secrets automatically on a schedule.'),
]

print(f'Total training pairs: {len(training_pairs)}')
print(f'\nDomain categories covered:')
categories = ['Kubernetes/Containers', 'Database Admin', 'Networking/Security', 
              'Monitoring/Observability', 'CI/CD Deployment', 'Infrastructure as Code']
for cat in categories:
    print(f'  - {cat}')

In [ ]:
# Create evaluation data (held out -- not used in training)
eval_queries = [
    'How do I horizontally autoscale my deployment?',
    'My pod is OOMKilled, what should I change?',
    'How to make database queries faster?',
    'Setting up database streaming replication',
    'How to renew my HTTPS certificate?',
    'Configure health checks for my load balancer',
    'How to set up Prometheus alerts?',
    'Setting up distributed tracing across microservices',
    'How to implement canary releases?',
    'Managing Terraform remote state',
]

# Ground truth: index into training_pairs documents (0-indexed)
eval_ground_truth = [0, 2, 6, 8, 11, 13, 16, 17, 21, 25]

# Extract all documents for retrieval
all_documents = [pair[1] for pair in training_pairs]

print(f'Evaluation queries: {len(eval_queries)}')
print(f'Document corpus: {len(all_documents)}')
print(f'\nSample evaluation pair:')
print(f'  Query: "{eval_queries[0]}"')
print(f'  Expected doc idx: {eval_ground_truth[0]}')
print(f'  Expected doc: "{all_documents[eval_ground_truth[0]][:80]}..."')

### 1.2 Data Quality Assessment

Before training, it is worth examining the quality and diversity of our training data. Effective
fine-tuning requires pairs that are semantically meaningful and cover the diversity of queries we
expect in production. If the training data clusters too tightly around a few topics, the model
will overfit to those topics and underperform on others.

We can use the base model itself to audit the data: embed all queries and documents, check their
distribution, and identify any gaps or redundancies.

In [ ]:
# Load base model and assess training data
base_model = SentenceTransformer('all-MiniLM-L6-v2')

train_queries = [pair[0] for pair in training_pairs]
train_docs = [pair[1] for pair in training_pairs]

q_emb = base_model.encode(train_queries)
d_emb = base_model.encode(train_docs)

# Check diversity: average pairwise similarity among queries
q_sims = cosine_similarity(q_emb)
mask = ~np.eye(len(q_sims), dtype=bool)
avg_q_sim = q_sims[mask].mean()
max_q_sim = q_sims[mask].max()
max_idx = np.unravel_index(np.argmax(q_sims * (~np.eye(len(q_sims), dtype=bool))), q_sims.shape)

print(f'Training data diversity analysis:')
print(f'  Average query-query similarity: {avg_q_sim:.3f}')
print(f'  Maximum query-query similarity: {max_q_sim:.3f}')
print(f'    Most similar queries:')
print(f'    "{train_queries[max_idx[0]]}"')
print(f'    "{train_queries[max_idx[1]]}"')

# Check query-document alignment
qd_sims = cosine_similarity(q_emb, d_emb)
diagonal_sims = np.diag(qd_sims)
print(f'\n  Average query-to-own-document similarity: {diagonal_sims.mean():.3f}')
print(f'  Min query-to-own-document similarity: {diagonal_sims.min():.3f}')
weakest = diagonal_sims.argmin()
print(f'    Weakest pair: "{train_queries[weakest]}" <-> doc {weakest} (sim={diagonal_sims[weakest]:.3f})')

---
## 2. Fine-Tune Embedding Model

### 2.1 Training Configuration

We fine-tune `all-MiniLM-L6-v2` using Multiple Negatives Ranking Loss (MNRL). This loss function
treats every other document in the batch as a negative for each query, which means our effective
number of negatives scales with batch size. With a batch size of 16, each query sees 15 in-batch
negatives per step.

Key hyperparameters:
- **Learning rate**: 2e-5 is standard for sentence-transformer fine-tuning. Higher rates risk
  catastrophic forgetting; lower rates require more epochs.
- **Epochs**: With a small dataset, we train for more epochs (20) but monitor for overfitting.
- **Warmup**: 10% of training steps, following the standard transformer warmup schedule.
- **Batch size**: Larger is better for MNRL (more negatives), but limited by GPU memory.

We also set up an evaluation callback that runs after each epoch, tracking retrieval accuracy
on our held-out queries. This lets us detect overfitting and select the best checkpoint.

In [ ]:
# Prepare training data
train_examples = [InputExample(texts=[q, d]) for q, d in training_pairs]
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

# Create evaluation callback
# InformationRetrievalEvaluator needs specific format
ir_queries = {str(i): q for i, q in enumerate(eval_queries)}
ir_corpus = {str(i): d for i, d in enumerate(all_documents)}
ir_relevant = {str(i): {str(gt)} for i, gt in enumerate(eval_ground_truth)}

ir_evaluator = evaluation.InformationRetrievalEvaluator(
    queries=ir_queries,
    corpus=ir_corpus,
    relevant_docs=ir_relevant,
    name='tech-docs-eval',
    show_progress_bar=False,
)

print(f'Training examples: {len(train_examples)}')
print(f'Batch size: 16')
print(f'Steps per epoch: {len(train_dataloader)}')
print(f'Evaluation queries: {len(eval_queries)}')

In [ ]:
# Evaluate base model first (pre fine-tuning baseline)
print('Base model evaluation on retrieval task:')
base_score = ir_evaluator(base_model)
print(f'  IR evaluation score: {base_score:.4f}')

In [ ]:
# Fine-tune
ft_model = SentenceTransformer('all-MiniLM-L6-v2')
train_loss = losses.MultipleNegativesRankingLoss(ft_model)

num_epochs = 20
warmup_steps = int(len(train_dataloader) * num_epochs * 0.1)

print(f'Starting fine-tuning for {num_epochs} epochs ({warmup_steps} warmup steps)...')
start_time = time.time()

ft_model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=num_epochs,
    warmup_steps=warmup_steps,
    evaluator=ir_evaluator,
    evaluation_steps=len(train_dataloader),  # evaluate every epoch
    show_progress_bar=True,
    output_path=None,
)

train_time = time.time() - start_time
print(f'\nFine-tuning complete in {train_time:.1f}s')

In [ ]:
# Post fine-tuning evaluation
print('Fine-tuned model evaluation on retrieval task:')
ft_score = ir_evaluator(ft_model)
print(f'  IR evaluation score: {ft_score:.4f}')
print(f'  Improvement over base: {ft_score - base_score:+.4f}')

---
## 3. Evaluate on Standard Tasks

### 3.1 Retrieval Evaluation

Retrieval is the primary use case for our fine-tuned model. We evaluate using standard information
retrieval metrics: Recall@k (what fraction of relevant documents appear in the top k results),
Mean Reciprocal Rank (the average of 1/rank for the first relevant result), and NDCG (normalized
discounted cumulative gain, which accounts for the position of relevant results).

We compare the base and fine-tuned models side by side to quantify the impact of fine-tuning.

In [ ]:
def evaluate_retrieval_detailed(model, queries, documents, ground_truth, model_name='Model'):
    """
    Compute detailed retrieval metrics.
    Returns dict of metrics and the similarity matrix.
    """
    q_emb = model.encode(queries)
    d_emb = model.encode(documents)
    sims = cosine_similarity(q_emb, d_emb)
    
    metrics = {'recall@1': 0, 'recall@3': 0, 'recall@5': 0, 'mrr': 0}
    
    for i, gt_idx in enumerate(ground_truth):
        ranked = np.argsort(sims[i])[::-1]
        rank = np.where(ranked == gt_idx)[0][0] + 1  # 1-indexed
        
        metrics['mrr'] += 1.0 / rank
        for k in [1, 3, 5]:
            if gt_idx in ranked[:k]:
                metrics[f'recall@{k}'] += 1
    
    n = len(queries)
    for key in metrics:
        metrics[key] /= n
    
    return metrics, sims

base_metrics, base_sims = evaluate_retrieval_detailed(
    base_model, eval_queries, all_documents, eval_ground_truth, 'Base'
)
ft_metrics, ft_sims = evaluate_retrieval_detailed(
    ft_model, eval_queries, all_documents, eval_ground_truth, 'Fine-Tuned'
)

print(f'{"Metric":<12} {"Base":>10} {"Fine-Tuned":>12} {"Delta":>10}')
print('-' * 47)
for key in ['recall@1', 'recall@3', 'recall@5', 'mrr']:
    bv = base_metrics[key]
    fv = ft_metrics[key]
    delta = fv - bv
    print(f'{key:<12} {bv:>10.1%} {fv:>12.1%} {delta:>+10.1%}')

In [ ]:
# Visualize retrieval: per-query rank comparison
def get_ranks(sims, ground_truth):
    ranks = []
    for i, gt_idx in enumerate(ground_truth):
        ranked = np.argsort(sims[i])[::-1]
        rank = np.where(ranked == gt_idx)[0][0] + 1
        ranks.append(rank)
    return ranks

base_ranks = get_ranks(base_sims, eval_ground_truth)
ft_ranks = get_ranks(ft_sims, eval_ground_truth)

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(eval_queries))
width = 0.35

bars1 = ax.bar(x - width/2, base_ranks, width, label='Base Model', color='#F44336', alpha=0.8)
bars2 = ax.bar(x + width/2, ft_ranks, width, label='Fine-Tuned', color='#2196F3', alpha=0.8)

ax.set_xlabel('Query Index', fontsize=12)
ax.set_ylabel('Rank of Correct Document (lower is better)', fontsize=12)
ax.set_title('Retrieval Rank: Base vs Fine-Tuned Model', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels([f'Q{i}' for i in range(len(eval_queries))], fontsize=10)
ax.legend(fontsize=11)
ax.axhline(y=1, color='green', linestyle='--', alpha=0.5, label='Perfect rank')
ax.invert_yaxis()  # Lower rank is better, show at top

plt.tight_layout()
plt.show()
print('Lower bars indicate better retrieval. Rank 1 means the correct document was the top result.')

### 3.2 Semantic Textual Similarity (STS) Evaluation

STS measures how well the model's similarity scores correlate with human judgments. We create a
small STS evaluation set with pairs of sentences and human similarity scores (0-5 scale). The
evaluation metric is Spearman rank correlation between model similarities and human scores.

A good embedding model should produce similarity scores that are monotonically related to human
judgments -- sentences that humans rate as more similar should have higher cosine similarity.

In [ ]:
# Create domain-specific STS evaluation pairs
sts_pairs = [
    ('kubernetes pod autoscaling', 'automatically scale containers based on CPU usage', 4.5),
    ('database index optimization', 'improving query performance with B-tree indexes', 4.0),
    ('SSL certificate setup', 'configuring HTTPS encryption for web server', 4.2),
    ('kubernetes pod autoscaling', 'database backup procedures', 0.5),
    ('load balancer configuration', 'traffic distribution across server instances', 4.3),
    ('monitoring alerting rules', 'sending notifications when metrics exceed thresholds', 4.1),
    ('CI pipeline optimization', 'making builds and tests run faster', 3.8),
    ('terraform state management', 'the weather forecast for next week', 0.1),
    ('container image scanning', 'checking Docker images for security vulnerabilities', 4.4),
    ('git branching strategy', 'version control workflow with feature branches', 4.0),
    ('network firewall rules', 'controlling inbound and outbound traffic', 4.2),
    ('database migration tools', 'schema version control and upgrade scripts', 3.9),
    ('log aggregation', 'collecting and centralizing application logs', 4.5),
    ('canary deployment', 'gradually rolling out changes to a subset of users', 4.3),
    ('kubernetes pod autoscaling', 'recipe for chocolate cake', 0.0),
]

sts_sent1 = [p[0] for p in sts_pairs]
sts_sent2 = [p[1] for p in sts_pairs]
sts_scores = [p[2] for p in sts_pairs]

def evaluate_sts(model, sent1, sent2, human_scores, name='Model'):
    emb1 = model.encode(sent1)
    emb2 = model.encode(sent2)
    model_scores = [cosine_similarity([e1], [e2])[0][0] for e1, e2 in zip(emb1, emb2)]
    correlation, p_value = spearmanr(human_scores, model_scores)
    return correlation, p_value, model_scores

base_corr, base_p, base_sts_scores = evaluate_sts(base_model, sts_sent1, sts_sent2, sts_scores, 'Base')
ft_corr, ft_p, ft_sts_scores = evaluate_sts(ft_model, sts_sent1, sts_sent2, sts_scores, 'Fine-Tuned')

print(f'STS Evaluation (Spearman correlation):')
print(f'  Base model:       rho={base_corr:.4f} (p={base_p:.4f})')
print(f'  Fine-tuned model: rho={ft_corr:.4f} (p={ft_p:.4f})')

In [ ]:
# Visualize STS correlation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, scores, corr, name, color in zip(
    axes,
    [base_sts_scores, ft_sts_scores],
    [base_corr, ft_corr],
    ['Base Model', 'Fine-Tuned Model'],
    ['#F44336', '#2196F3']
):
    ax.scatter(sts_scores, scores, c=color, s=80, alpha=0.7, edgecolors='white', linewidth=0.5)
    
    # Trend line
    z = np.polyfit(sts_scores, scores, 1)
    p = np.poly1d(z)
    x_line = np.linspace(0, 5, 100)
    ax.plot(x_line, p(x_line), '--', color=color, alpha=0.5)
    
    ax.set_xlabel('Human Similarity Score (0-5)', fontsize=11)
    ax.set_ylabel('Model Cosine Similarity', fontsize=11)
    ax.set_title(f'{name} (Spearman rho={corr:.3f})', fontsize=13)
    ax.set_xlim(-0.3, 5.3)

plt.suptitle('Semantic Textual Similarity: Human vs Model Scores', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 3.3 Clustering Evaluation

Clustering quality tells us whether the embedding space has meaningful structure at the topic level.
We embed all documents and run K-Means clustering, then compare the clusters against our known
topic categories using metrics like Adjusted Rand Index (ARI), Normalized Mutual Information (NMI),
and silhouette score.

Good embeddings should naturally group documents by topic, even without explicit clustering training.
Fine-tuning should improve this grouping by sharpening the distinction between domains.

In [ ]:
# Assign topic labels to documents
topic_labels = (
    [0]*6 +   # Kubernetes
    [1]*5 +   # Database
    [2]*5 +   # Networking/Security
    [3]*4 +   # Monitoring
    [4]*5 +   # CI/CD
    [5]*3     # IaC
)
topic_names = ['Kubernetes', 'Database', 'Network/Sec', 'Monitoring', 'CI/CD', 'IaC']

n_clusters = len(topic_names)

def evaluate_clustering(model, documents, true_labels, n_clusters, name='Model'):
    emb = model.encode(documents)
    
    km = KMeans(n_clusters=n_clusters, random_state=SEED, n_init=10)
    pred_labels = km.fit_predict(emb)
    
    ari = adjusted_rand_score(true_labels, pred_labels)
    nmi = normalized_mutual_info_score(true_labels, pred_labels)
    v_measure = v_measure_score(true_labels, pred_labels)
    silhouette = silhouette_score(emb, pred_labels)
    
    return {
        'ARI': ari,
        'NMI': nmi,
        'V-measure': v_measure,
        'Silhouette': silhouette,
    }, emb

base_cluster, base_doc_emb = evaluate_clustering(base_model, all_documents, topic_labels, n_clusters, 'Base')
ft_cluster, ft_doc_emb = evaluate_clustering(ft_model, all_documents, topic_labels, n_clusters, 'Fine-Tuned')

print(f'{"Metric":<12} {"Base":>10} {"Fine-Tuned":>12} {"Delta":>10}')
print('-' * 47)
for key in ['ARI', 'NMI', 'V-measure', 'Silhouette']:
    bv = base_cluster[key]
    fv = ft_cluster[key]
    print(f'{key:<12} {bv:>10.3f} {fv:>12.3f} {fv-bv:>+10.3f}')

In [ ]:
# t-SNE visualization: before and after fine-tuning
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

colors_map = ['#E53935', '#1E88E5', '#43A047', '#FB8C00', '#8E24AA', '#00ACC1']

for ax, emb, title in zip(axes, [base_doc_emb, ft_doc_emb], ['Base Model', 'Fine-Tuned Model']):
    tsne = TSNE(n_components=2, random_state=SEED, perplexity=8)
    proj = tsne.fit_transform(emb)
    
    for topic_idx, topic_name in enumerate(topic_names):
        mask = np.array(topic_labels) == topic_idx
        ax.scatter(
            proj[mask, 0], proj[mask, 1],
            c=colors_map[topic_idx], s=100, label=topic_name,
            alpha=0.8, edgecolors='white', linewidth=0.5
        )
    
    ax.set_title(f'{title} -- Document Embedding Space', fontsize=13)
    ax.legend(fontsize=9, loc='best')
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('t-SNE Visualization: Topic Clustering Before and After Fine-Tuning', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
print('Fine-tuning should produce tighter, more separated topic clusters.')

### 3.4 Comprehensive Evaluation Summary

Let us compile all evaluation results into a single comparison table. A good fine-tuned model should
improve retrieval metrics (its primary training objective) without significantly degrading STS
correlation or clustering quality.

In [ ]:
# Comprehensive comparison
print('=' * 60)
print('COMPREHENSIVE EVALUATION: Base vs Fine-Tuned')
print('=' * 60)

print('\n--- Retrieval ---')
for key in ['recall@1', 'recall@3', 'recall@5', 'mrr']:
    print(f'  {key:<12} Base: {base_metrics[key]:.1%}  FT: {ft_metrics[key]:.1%}  ({ft_metrics[key]-base_metrics[key]:+.1%})')

print('\n--- Semantic Textual Similarity ---')
print(f'  {"Spearman rho":<12} Base: {base_corr:.3f}  FT: {ft_corr:.3f}  ({ft_corr-base_corr:+.3f})')

print('\n--- Clustering ---')
for key in ['ARI', 'NMI', 'V-measure', 'Silhouette']:
    print(f'  {key:<12} Base: {base_cluster[key]:.3f}  FT: {ft_cluster[key]:.3f}  ({ft_cluster[key]-base_cluster[key]:+.3f})')

print('\n--- Summary ---')
retrieval_improved = ft_metrics['mrr'] > base_metrics['mrr']
sts_maintained = ft_corr > base_corr - 0.1
cluster_maintained = ft_cluster['NMI'] > base_cluster['NMI'] - 0.1
print(f'  Retrieval improved: {retrieval_improved}')
print(f'  STS maintained (within 0.1): {sts_maintained}')
print(f'  Clustering maintained (within 0.1): {cluster_maintained}')

---
## 4. Quantization and Deployment

### 4.1 Scalar Quantization (float32 to int8)

For production deployment, we quantize the fine-tuned model's embeddings from float32 to int8.
This reduces storage by 4x and often improves inference speed due to better cache utilization.
As we saw in the expert notebook, scalar quantization preserves cosine similarity with negligible
quality loss.

The quantization pipeline:
1. Encode a calibration set to determine per-dimension min/max values.
2. Map each float32 value to the nearest uint8 level.
3. Store the calibration parameters alongside the quantized embeddings.
4. At query time, quantize the query using the same calibration and compute similarity on int8 values.

In [ ]:
def scalar_quantize_int8(embeddings: np.ndarray) -> tuple:
    """
    Quantize float32 embeddings to uint8 with per-dimension calibration.
    Returns (quantized_embeddings, calibration_params).
    """
    mins = embeddings.min(axis=0)
    maxs = embeddings.max(axis=0)
    ranges = maxs - mins
    ranges[ranges == 0] = 1.0
    
    normalized = (embeddings - mins) / ranges
    quantized = np.clip(np.round(normalized * 255), 0, 255).astype(np.uint8)
    
    calibration = {'mins': mins, 'ranges': ranges}
    return quantized, calibration


def quantized_cosine_similarity(q_quant, d_quant):
    """Compute approximate cosine similarity on quantized embeddings."""
    q_f = q_quant.astype(np.float32)
    d_f = d_quant.astype(np.float32)
    return cosine_similarity(q_f, d_f)


# Quantize document embeddings
ft_doc_emb_full = ft_model.encode(all_documents)
doc_quant, calibration = scalar_quantize_int8(ft_doc_emb_full)

print(f'Original embedding dtype: {ft_doc_emb_full.dtype}, shape: {ft_doc_emb_full.shape}')
print(f'Quantized embedding dtype: {doc_quant.dtype}, shape: {doc_quant.shape}')
print(f'Storage per embedding: {ft_doc_emb_full.shape[1] * 4} bytes (fp32) -> {doc_quant.shape[1]} bytes (uint8)')
print(f'Compression ratio: {ft_doc_emb_full.shape[1] * 4 / doc_quant.shape[1]:.1f}x')

In [ ]:
# Benchmark quantized retrieval quality
q_emb_full = ft_model.encode(eval_queries)
q_quant, _ = scalar_quantize_int8(q_emb_full)

# Full precision similarity
fp32_sims = cosine_similarity(q_emb_full, ft_doc_emb_full)

# Quantized similarity
int8_sims = quantized_cosine_similarity(q_quant, doc_quant)

# Compare rankings
def recall_at_k(query_sims, gt_sims, ground_truth, k):
    correct = 0
    for i, gt_idx in enumerate(ground_truth):
        gt_topk = set(np.argsort(gt_sims[i])[-k:])
        pred_topk = set(np.argsort(query_sims[i])[-k:])
        if gt_idx in pred_topk:
            correct += 1
    return correct / len(ground_truth)

print('Quantization Quality Benchmark:')
print(f'{"Metric":<15} {"FP32":>8} {"INT8":>8} {"Match":>8}')
print('-' * 40)
for k in [1, 3, 5]:
    fp32_r = recall_at_k(fp32_sims, fp32_sims, eval_ground_truth, k)
    int8_r = recall_at_k(int8_sims, fp32_sims, eval_ground_truth, k)
    match = 'Yes' if abs(fp32_r - int8_r) < 0.01 else 'No'
    print(f'Recall@{k:<8} {fp32_r:>8.1%} {int8_r:>8.1%} {match:>8}')

# Similarity correlation
corr, _ = spearmanr(fp32_sims.flatten(), int8_sims.flatten())
print(f'\nSimilarity rank correlation (Spearman): {corr:.4f}')

In [ ]:
# Visualize quantization fidelity
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: FP32 vs INT8 similarities
ax1.scatter(fp32_sims.flatten(), int8_sims.flatten(), alpha=0.5, s=20, color='#3F51B5')
ax1.plot([0, 1], [0, 1], 'r--', alpha=0.5)
ax1.set_xlabel('FP32 Cosine Similarity', fontsize=11)
ax1.set_ylabel('INT8 Cosine Similarity', fontsize=11)
ax1.set_title(f'Quantization Fidelity (rho={corr:.4f})', fontsize=13)

# Histogram of similarity errors
errors = (fp32_sims - int8_sims).flatten()
ax2.hist(errors, bins=50, color='#FF9800', alpha=0.8, edgecolor='white')
ax2.axvline(x=0, color='red', linestyle='--', alpha=0.7)
ax2.set_xlabel('Similarity Error (FP32 - INT8)', fontsize=11)
ax2.set_ylabel('Count', fontsize=11)
ax2.set_title(f'Quantization Error Distribution (mean={errors.mean():.4f}, std={errors.std():.4f})', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# Speed benchmark: FP32 vs INT8 similarity computation
n_queries = 100
n_docs = 1000

# Generate larger test data
large_q = np.random.randn(n_queries, 384).astype(np.float32)
large_d = np.random.randn(n_docs, 384).astype(np.float32)
large_q_int8 = np.clip(np.round((large_q - large_q.min()) / (large_q.max() - large_q.min()) * 255), 0, 255).astype(np.uint8)
large_d_int8 = np.clip(np.round((large_d - large_d.min()) / (large_d.max() - large_d.min()) * 255), 0, 255).astype(np.uint8)

# FP32 timing
times_fp32 = []
for _ in range(20):
    start = time.perf_counter()
    _ = cosine_similarity(large_q, large_d)
    times_fp32.append(time.perf_counter() - start)

# INT8 timing
times_int8 = []
for _ in range(20):
    start = time.perf_counter()
    _ = quantized_cosine_similarity(large_q_int8, large_d_int8)
    times_int8.append(time.perf_counter() - start)

avg_fp32 = np.mean(times_fp32) * 1000
avg_int8 = np.mean(times_int8) * 1000

print(f'Similarity computation ({n_queries} queries x {n_docs} docs):')
print(f'  FP32: {avg_fp32:.2f} ms')
print(f'  INT8: {avg_int8:.2f} ms')
print(f'  Note: INT8 requires float cast for numpy cosine_similarity.')
print(f'  True INT8 SIMD operations (in FAISS/USearch) are significantly faster.')

### 4.2 Model Card

A model card documents the model's capabilities, limitations, training data, and evaluation results.
It is essential for responsible deployment and helps downstream users understand what the model can
and cannot do. Below we generate a structured model card from our evaluation results.

In [ ]:
model_card = {
    'model_name': 'tech-docs-MiniLM-L6-v2-finetuned',
    'base_model': 'all-MiniLM-L6-v2',
    'description': 'Sentence embedding model fine-tuned for technical infrastructure documentation retrieval.',
    'embedding_dimension': 384,
    'max_sequence_length': 256,
    'training': {
        'loss_function': 'MultipleNegativesRankingLoss',
        'num_training_pairs': len(training_pairs),
        'epochs': num_epochs,
        'batch_size': 16,
        'training_time_seconds': round(train_time, 1),
    },
    'evaluation': {
        'retrieval': {k: round(v, 3) for k, v in ft_metrics.items()},
        'sts_spearman_correlation': round(ft_corr, 3),
        'clustering': {k: round(v, 3) for k, v in ft_cluster.items()},
        'improvement_over_base': {
            'retrieval_mrr': round(ft_metrics['mrr'] - base_metrics['mrr'], 3),
            'sts_correlation': round(ft_corr - base_corr, 3),
            'clustering_nmi': round(ft_cluster['NMI'] - base_cluster['NMI'], 3),
        },
    },
    'quantization': {
        'int8_supported': True,
        'int8_similarity_correlation': round(corr, 4),
        'storage_per_embedding_bytes': {
            'fp32': 384 * 4,
            'int8': 384,
        },
    },
    'intended_use': 'Semantic search over technical infrastructure documentation, runbooks, and configuration guides.',
    'limitations': [
        'Fine-tuned on a small dataset; may not generalize to all technical domains.',
        'Not tested on non-English text.',
        'May lose some general-domain capability compared to the base model.',
    ],
    'domain_categories': [
        'Kubernetes / Container Orchestration',
        'Database Administration',
        'Networking and Security',
        'Monitoring and Observability',
        'CI/CD and Deployment',
        'Infrastructure as Code',
    ],
}

print(json.dumps(model_card, indent=2))

In [ ]:
# Final summary visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Retrieval metrics comparison
metrics_keys = ['recall@1', 'recall@3', 'recall@5', 'mrr']
x = np.arange(len(metrics_keys))
width = 0.35
axes[0].bar(x - width/2, [base_metrics[k] for k in metrics_keys], width,
            label='Base', color='#F44336', alpha=0.8)
axes[0].bar(x + width/2, [ft_metrics[k] for k in metrics_keys], width,
            label='Fine-Tuned', color='#2196F3', alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(['R@1', 'R@3', 'R@5', 'MRR'])
axes[0].set_title('Retrieval Metrics', fontsize=13)
axes[0].set_ylim(0, 1.1)
axes[0].legend(fontsize=10)

# 2. Clustering metrics comparison
cluster_keys = ['ARI', 'NMI', 'V-measure', 'Silhouette']
x2 = np.arange(len(cluster_keys))
axes[1].bar(x2 - width/2, [base_cluster[k] for k in cluster_keys], width,
            label='Base', color='#F44336', alpha=0.8)
axes[1].bar(x2 + width/2, [ft_cluster[k] for k in cluster_keys], width,
            label='Fine-Tuned', color='#2196F3', alpha=0.8)
axes[1].set_xticks(x2)
axes[1].set_xticklabels(cluster_keys)
axes[1].set_title('Clustering Metrics', fontsize=13)
axes[1].legend(fontsize=10)

# 3. Storage comparison
storage_labels = ['FP32', 'INT8']
storage_vals = [384 * 4 / 1024, 384 / 1024]  # KB per embedding
bars = axes[2].bar(storage_labels, storage_vals, color=['#F44336', '#4CAF50'], alpha=0.8)
axes[2].set_ylabel('KB per embedding')
axes[2].set_title('Storage Requirements', fontsize=13)
for bar, val in zip(bars, storage_vals):
    axes[2].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                 f'{val:.2f} KB', ha='center', fontsize=11, fontweight='bold')

plt.suptitle('Build Project: Complete Evaluation Summary', fontsize=15, y=1.03)
plt.tight_layout()
plt.show()

---
## Summary

In this build notebook we completed a full embedding model development cycle:

1. **Domain and Data**: We defined a technical documentation domain with 28 query-document training
   pairs across 6 infrastructure categories. We analyzed the training data for diversity and
   quality using the base model.

2. **Fine-Tuning**: We fine-tuned `all-MiniLM-L6-v2` with MNRL loss for 20 epochs, using an
   information retrieval evaluator as a training callback.

3. **Evaluation**: We evaluated on three complementary tasks -- retrieval (Recall@k, MRR),
   semantic textual similarity (Spearman correlation), and clustering (ARI, NMI, V-measure,
   silhouette). We visualized the embedding space with t-SNE before and after fine-tuning.

4. **Quantization and Deployment**: We quantized embeddings to INT8, verified quality preservation
   with similarity rank correlation, benchmarked speed, and generated a model card.

This pattern -- domain analysis, data curation, training, multi-task evaluation, quantization,
and documentation -- is the production workflow for embedding model development.

---

**Next**: `../03-attention-mechanisms/foundations.ipynb` -- Dive into the attention mechanism
that powers the transformer models generating these embeddings.